# Participatory Analytics - Visualisation results of the sketchmap tool

This notebook demonstrates how to visualise a geojson file using well known libraries and the results from the digitalisation through [the Sketchmap tool](https://sketchmaptool.org).

We use a dummy geojson file that can be replaced with the result of the digitalisation.




[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/urbanbigdatacentre/Advanced-topics-UA-2026/blob/main/visual.ipynb)

## Running this Notebook in Google Colab

You can run this notebook directly in your browser using **Google Colab** — no local installation required.

### Steps

1. Click the **"Open in Colab"** badge above.
2. Once the notebook opens in Colab, click **Runtime → Run all** (or press `Ctrl+F9`) to execute all cells.
3. The notebook will automatically download the `dummy_polygon.geojson` file if it is not found locally.

```markdown
### Alternative: Open Directly from GitHub

You can also access the notebook directly from the repository:

- **Repository:** [Advanced-topics-UA-2026](https://github.com/urbanbigdatacentre/Advanced-topics-UA-2026)
- **Notebook:** [visual.ipynb](https://github.com/urbanbigdatacentre/Advanced-topics-UA-2026/blob/main/visual.ipynb)

To open in Colab from the repository:
1. Navigate to the notebook file in GitHub
2. Click the **"Open in Colab"** option (if available) or copy the notebook URL
3. Go to [colab.research.google.com](https://colab.research.google.com)
4. Select **File → Open notebook** and paste the GitHub URL
5. Click **Runtime → Run all** to execute
```

### Requirements

No additional setup is needed. The notebook handles missing files automatically and all required libraries (`geopandas`, `folium`) will be installed if not already available in the Colab environment.

> 💡 **Tip:** If you encounter a `ModuleNotFoundError`, add and run the following cell before executing the rest of the notebook:
> ```python
> !pip install geopandas folium
> ```

## Task 1: Visualise a GeoJSON file on a map

Here you have an example of reading a GeoJSON file and visualising it on a map. Your task is to visualise the results of the digitalisation through the Sketchmap tool, which you can download as a GeoJSON file and replace the dummy file used in this notebook.

In [ ]:
from pathlib import Path
import urllib.request
import geopandas as gpd
import folium

# The following line sets the pat to access a GeoJSON File. Once you have processed the data through the Sketchmap tool, 
# you can upload the file to google colab and replace the dummy_polygon.geojson with the path to your file.
geojson_path = Path("dummy_polygon.geojson")
if not geojson_path.exists():
    raw_url = "https://raw.githubusercontent.com/urbanbigdatacentre/Advanced-topics-UA-2026/main/dummy_polygon.geojson"
    print(f"{geojson_path} not found locally. Downloading from: {raw_url}")
    urllib.request.urlretrieve(raw_url, geojson_path)

# Read the GeoJSON file and project to WGS84
gdf = gpd.read_file(geojson_path.as_posix())
gdf_wgs84 = gdf.to_crs(epsg=4326)

# Center map on polygon centroid
center = [
    gdf_wgs84.geometry.centroid.y.mean(),
    gdf_wgs84.geometry.centroid.x.mean(),
]

# Create interactive web map with OSM tiles
m = folium.Map(location=center, zoom_start=14, tiles="OpenStreetMap")

# Add polygon layer
folium.GeoJson(
    gdf_wgs84,
    name="dummy_polygon",
    style_function=lambda x: {"color": "red", "weight": 3, "fillOpacity": 0},
).add_to(m)

# Zoom to data bounds
bounds = gdf_wgs84.total_bounds  # [minx, miny, maxx, maxy]
m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])

display(m)

## Task 2: Visualise a GeoTIFF file on a map
The sketchmap tool also allows you to download the results of the digitalisation as a GeoTIFF file. The following lines of code show how to read a GeoTIFF file and visualise it on a map. You can replace the dummy GeoTIFF file with the one you have downloaded from the sketchmap tool.

In [ ]:
import numpy as np
import rasterio
from rasterio.warp import transform_bounds
from rasterio.io import MemoryFile
import base64
from PIL import Image
import io

# Resolve GeoTIFF path for local runs and cloud runtimes (e.g., Colab)
geotiff_path = Path("YOUR_FILE_NAME.tif")  # Replace with your GeoTIFF file path
# Uncomment below to download from a remote URL if not found locally
# if not geotiff_path.exists():
#     raw_url = "https://your-url/your_file.tif"
#     print(f"{geotiff_path} not found locally. Downloading from: {raw_url}")
#     urllib.request.urlretrieve(raw_url, geotiff_path)

with rasterio.open(geotiff_path) as src:
    # Reproject bounds to WGS84
    bounds_wgs84 = transform_bounds(src.crs, "EPSG:4326", *src.bounds)
    minx, miny, maxx, maxy = bounds_wgs84

    # Read and normalise the first 3 bands (RGB) or single band
    if src.count >= 3:
        data = src.read([1, 2, 3])  # RGB
    else:
        band = src.read(1)
        data = np.stack([band, band, band])  # Grayscale → RGB

    # Normalise to 0–255
    data = data.astype(float)
    for i in range(3):
        band_min, band_max = np.nanmin(data[i]), np.nanmax(data[i])
        if band_max > band_min:
            data[i] = (data[i] - band_min) / (band_max - band_min) * 255
    data = data.astype(np.uint8)

    # Convert to RGBA PNG in memory
    rgba = np.dstack((data[0], data[1], data[2], np.full(data[0].shape, 200, dtype=np.uint8)))
    img = Image.fromarray(rgba, mode="RGBA")
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    img_data_url = f"data:image/png;base64,{encoded}"

# Center map on GeoTIFF bounds
raster_center = [(miny + maxy) / 2, (minx + maxx) / 2]

# Create interactive web map
m_raster = folium.Map(location=raster_center, zoom_start=12, tiles="OpenStreetMap")

# Overlay the GeoTIFF as an image
folium.raster_layers.ImageOverlay(
    image=img_data_url,
    bounds=[[miny, minx], [maxy, maxx]],
    opacity=0.7,
    name="GeoTIFF Overlay",
).add_to(m_raster)

# Optionally overlay the polygon from the previous cell
folium.GeoJson(
    gdf_wgs84,
    name="dummy_polygon",
    style_function=lambda x: {"color": "red", "weight": 3, "fillOpacity": 0},
).add_to(m_raster)

folium.LayerControl().add_to(m_raster)
m_raster.fit_bounds([[miny, minx], [maxy, maxx]])

display(m_raster)